In [1]:
import os
import json
from tqdm import tqdm
import networkx as nx
from fastnode2vec import Node2Vec, Graph

In [2]:
with open('../data/open_alex_graph.json') as f:
    oalex_graph_dict = json.load(f)

In [3]:
with open("../data/pmid_to_openAlex.json") as f:
    pmid_to_openAlex = json.load(f)

In [4]:
with open("../data/pmid2node_id_sample.json", "r") as f:
    pmid2node_id = json.load(f)

In [5]:
openAlex_included = [v for k,v in tqdm(pmid_to_openAlex.items()) if k in pmid2node_id and v in oalex_graph_dict]

100%|███████████████████████████████████████| 3616600/3616600 [00:00<00:00, 7243548.16it/s]


In [6]:
openAlex_included

['https://openalex.org/W2415906703',
 'https://openalex.org/W2169870814',
 'https://openalex.org/W2769987977',
 'https://openalex.org/W2062924831',
 'https://openalex.org/W2055958688',
 'https://openalex.org/W2055951262',
 'https://openalex.org/W2063144963',
 'https://openalex.org/W2019834368',
 'https://openalex.org/W2123820544',
 'https://openalex.org/W2793124808',
 'https://openalex.org/W2044538917',
 'https://openalex.org/W1591407057',
 'https://openalex.org/W2409689976',
 'https://openalex.org/W1771692675',
 'https://openalex.org/W2527838725',
 'https://openalex.org/W2804412990',
 'https://openalex.org/W2884996236',
 'https://openalex.org/W2111247269',
 'https://openalex.org/W2883370059',
 'https://openalex.org/W2062432778',
 'https://openalex.org/W1757309072',
 'https://openalex.org/W2168378092',
 'https://openalex.org/W2848069243',
 'https://openalex.org/W2387448306',
 'https://openalex.org/W2617274523',
 'https://openalex.org/W2412936433',
 'https://openalex.org/W2116081018',
 

In [7]:
len(openAlex_included)

101222

In [8]:
oalex_graph_dict

{'https://openalex.org/W1986812796': ['https://openalex.org/W60487933',
  'https://openalex.org/W1509884519',
  'https://openalex.org/W1552605193',
  'https://openalex.org/W1575402497',
  'https://openalex.org/W1671332310',
  'https://openalex.org/W1730334876',
  'https://openalex.org/W1964022954',
  'https://openalex.org/W1971545212',
  'https://openalex.org/W1982787778',
  'https://openalex.org/W1998655933',
  'https://openalex.org/W1998823477',
  'https://openalex.org/W1998871738',
  'https://openalex.org/W2000836791',
  'https://openalex.org/W2002453811',
  'https://openalex.org/W2003224205',
  'https://openalex.org/W2019132331',
  'https://openalex.org/W2020375422',
  'https://openalex.org/W2026725758',
  'https://openalex.org/W2032911360',
  'https://openalex.org/W2041241532',
  'https://openalex.org/W2055878329',
  'https://openalex.org/W2058296840',
  'https://openalex.org/W2059352537',
  'https://openalex.org/W2077517298',
  'https://openalex.org/W2083794851',
  'https://opena

In [9]:
# Create a graph
# graph = nx.fast_gnp_random_graph(n=100, p=0.5)
# graph = nx.Graph(oalex_graph_dict)

In [11]:
graph_lst = []
for k in tqdm(openAlex_included):
    values = list(set(oalex_graph_dict[k])& set(openAlex_included))
    for v in values:
        graph_lst.append((k, v))

100%|█████████████████████████████████████████████| 101222/101222 [07:08<00:00, 236.08it/s]


In [13]:
# import pickle as pkl
# with open("../data/graph_lst.pkl", "wb") as f:
#     pkl.dump(graph_lst, f)

In [26]:
graph = Graph(graph_lst, directed=False, weighted=False)

Reading graph: 100%|████████████████████████████| 39051/39051 [00:00<00:00, 1744432.72it/s]


In [27]:
# Precompute probabilities and generate walks - **ON WINDOWS ONLY WORKS WITH workers=1**
node2vec = Node2Vec(graph, dim=10, walk_length=100, context=10, p=2.0, q=0.5, workers=5)  # Use temp_folder for big graphs


# Embed nodes
# model = node2vec.fit(window=10, min_count=1, batch_words=4)  # Any keywords acceptable by gensim.Word2Vec can be passed, `dimensions` and `workers` are automatically passed (from the Node2Vec constructor)

# Look for most similar nodes
# model.wv.most_similar('2')  # Output node names are always strings

In [28]:
node2vec.train(epochs=1)

Training: 100%|████████████████████████████████████| 35090/35090 [00:04<00:00, 7209.78it/s]


In [29]:
node2vec

In [30]:
MODEL_DIR = "../models"
EMBEDDING_FILENAME = os.path.join(MODEL_DIR, 'node_model_sample.pkl')

In [31]:
node2vec.wv.save_word2vec_format(EMBEDDING_FILENAME)
# node2vec.save(EMBEDDING_FILENAME)